# Retail Capstone: Python Cleaning
Goal: clean the 3 CSVs, test H1–H5, export the star schema for Power BI.


## 1. Setup
- Go into the capstone_retail folder (%cd)
- Load the 3 CSVs with pd.read_csv. Returns load as text (dtype=str) because the feed is messy
- Check: (29970, 24) (863, 8) (2538, 11)

In [36]:
#A
#How to begin the connection to google drive
from google.colab import drive

drive.mount("/content/drive")

%cd /content/drive/MyDrive/capstone_retail

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/capstone_retail


In [37]:
#B
# Files and location
import pandas as pd

sales = pd.read_csv("csv/sales_lines.csv")
returns_raw = pd.read_csv("csv/returns_feed.csv", dtype=str)
customers = pd.read_csv("csv/customers.csv", dtype={"phone": str})

print(sales.shape)
print(returns_raw.shape)
print(customers.shape)

(29970, 24)
(863, 8)
(2538, 11)


## 2. Completed orders only
- value_counts() = COUNTIF for every value
- Filter: sales[sales["order_status"] == "Completed"]
- Check: 1,053 cancelled dropped → 28,917 left

In [38]:
#1
print(sales["order_status"].value_counts())

order_status
Completed    28917
Cancelled     1053
Name: count, dtype: int64


In [39]:
#2
sales = sales[sales["order_status"] == "Completed"]

print(len(sales))

28917


## 3. Fix the price typos
- unit_price > list_price × 1.5 → 12 typos → divide by 10
- list_price > unit_price × 5 → 128 typos → divide by 10
- .loc[rows, column] = go to these cells and change them

In [40]:
#3
price_typo = sales["unit_price"] > sales["list_price"] * 1.5

print(price_typo.sum())

12


In [41]:
#4
sales.loc[price_typo, "unit_price"] = sales.loc[price_typo, "unit_price"] / 10

print((sales["unit_price"] > sales["list_price"] * 1.5).sum())

0


In [42]:
#5
list_typo = sales["list_price"] > sales["unit_price"] * 5

print(list_typo.sum())

sales.loc[list_typo, "list_price"] = sales.loc[list_typo, "list_price"] / 10

print((sales["list_price"] > sales["unit_price"] * 5).sum())

128
0


## 4. Money columns
- gross_sales = qty × unit_price
- net_revenue = gross_sales − discount
- line_cost = qty × unit_cost_at_sale
- Check: 2,516,810.27 / 2,345,974.25 / 1,812,142.64

In [43]:
#6
sales["gross_sales"] = sales["quantity"] * sales["unit_price"]
sales["net_revenue"] = sales["gross_sales"] - sales["discount_amount"]
sales["line_cost"] = sales["quantity"] * sales["unit_cost_at_sale"]

print(sales[["gross_sales", "net_revenue", "line_cost"]].sum().round(2))

gross_sales    2516810.27
net_revenue    2345974.25
line_cost      1812142.64
dtype: float64


## 5. Year, guests, join keys
- to_datetime → real dates, .dt.year → year (8,643 / 9,730 / 10,544)
- Blank customer → "GUEST" (3,542) so they aren't dropped
- order_key and sku_key = dashes removed, for matching returns

In [44]:
#7
sales["order_date"] = pd.to_datetime(sales["order_date"])
sales["order_year"] = sales["order_date"].dt.year

sales["customer_code"] = sales["customer_code"].fillna("GUEST")

print(sales["order_year"].value_counts().sort_index())
print((sales["customer_code"] == "GUEST").sum())

order_year
2023     8643
2024     9730
2025    10544
Name: count, dtype: int64
3542


In [45]:
#8
sales["order_key"] = sales["order_number"].str.replace("-", "")
sales["sku_key"] = sales["sku"].str.replace("-", "")

print(sales[["order_number", "order_key", "sku", "sku_key"]].head())

  order_number   order_key         sku    sku_key
0  ORD-0000001  ORD0000001  SKU-101500  SKU101500
1  ORD-0000001  ORD0000001  SKU-102208  SKU102208
2  ORD-0000001  ORD0000001  SKU-102783  SKU102783
4  ORD-0000003  ORD0000003  SKU-100502  SKU100502
5  ORD-0000003  ORD0000003  SKU-100810  SKU100810


## 6. Revenue by year
- groupby = a pivot table (year in rows, sums in Values)
- 2023: 707,009.75 | 2024: 773,011.81 | 2025: 865,952.69

In [46]:
#9
by_year = sales.groupby("order_year")[["net_revenue", "line_cost"]].sum().round(2)

print(by_year)

            net_revenue  line_cost
order_year                        
2023          707009.75  541398.35
2024          773011.81  593495.20
2025          865952.69  677249.09


In [47]:
#C
# check what column names exist

# print(sales.columns.tolist())

## 7. Clean returns
- Keys: strip + upper + remove "-" (like TRIM/UPPER/SUBSTITUTE)
- Drop blank refs → 860, drop duplicates → 820
- Qty: regex (\d+) grabs the first number
- Refund: delete everything except digits and dots → 8 N/A left blank
- Merge to sales: 804 matched, 16 orphans dropped
- Fill the 8 refunds: net_revenue / quantity × qty_returned → total 75,767.96
- Quality reasons: 144 | Dates: format="mixed", 0 failed

In [48]:
#10
returns = returns_raw.copy()

returns["order_key"] = returns["order_number_text"].str.strip().str.upper().str.replace("-", "")
returns["sku_key"] = returns["sku_text"].str.strip().str.upper().str.replace("-", "")

print(returns[["order_number_text", "order_key", "sku_text", "sku_key"]].head())

  order_number_text   order_key     sku_text    sku_key
0        ORD0008707  ORD0008707   SKU-103113  SKU103113
1      ORD-0003042   ORD0003042   sku-100835  SKU100835
2      ORD-0011325   ORD0011325   SKU-102984  SKU102984
3       ORD-0007725  ORD0007725  SKU-101181   SKU101181
4      ORD-0011264   ORD0011264    SKU102061  SKU102061


In [49]:
#11
returns = returns.dropna(subset=["return_ref"])
print(len(returns))

returns = returns.drop_duplicates(subset=["return_ref"])
print(len(returns))

860
820


In [50]:
#12
returns["qty_returned"] = returns["quantity_text"].str.extract(r"(\d+)")[0].astype(int)

print(returns["quantity_text"].value_counts().head(10))
print(returns["qty_returned"].value_counts())

quantity_text
1         292
1 unit    169
1.0       164
2          86
2.0        37
2 unit     22
3.0        16
3          13
4           9
3 unit      7
Name: count, dtype: int64
qty_returned
1    625
2    145
3     36
4     14
Name: count, dtype: int64


In [51]:
#13
refund_clean = returns["refund_amount_text"].str.replace(r"[^0-9.]", "", regex=True)
returns["refund"] = pd.to_numeric(refund_clean, errors="coerce")

print(returns["refund"].isna().sum())
print(returns["refund"].sum().round(2))

8
76506.52


In [52]:
#14
returns = returns.merge(
    sales[["order_key", "sku_key", "order_number", "order_date", "store_code", "sku",
           "brand", "net_revenue", "quantity", "order_year", "category_name", "region_name"]],
    on=["order_key", "sku_key"],
    how="left",
    indicator=True,
)

print(returns["_merge"].value_counts())


_merge
both          804
left_only      16
right_only      0
Name: count, dtype: int64


In [53]:
#15
returns = returns[returns["_merge"] == "both"]
print(len(returns))

missing = returns["refund"].isna()
print(missing.sum())

returns.loc[missing, "refund"] = returns["net_revenue"] / returns["quantity"] * returns["qty_returned"]

print(returns["refund"].sum().round(2))

804
8
75767.96


In [54]:
#16
import numpy as np

is_quality = returns["return_reason"].str.contains("defect|stop|qual|turn on|broke", case=False)
returns["reason_group"] = np.where(is_quality, "Quality", "Other")

print(returns["reason_group"].value_counts())

reason_group
Other      660
Quality    144
Name: count, dtype: int64


In [55]:
#17
returns["return_date"] = pd.to_datetime(returns["return_date_text"].str.strip(), format="mixed")

print(returns["return_date"].isna().sum())
print(returns["return_date"].min(), returns["return_date"].max())

0
2023-01-08 00:00:00 2025-12-31 00:00:00


## 8. Clean customers
- City: strip + title + alias dictionary → 120 spellings become 30
- Email: regex pattern → 2,298 OK / 219 Blank / 21 Invalid
- Phone: digits only, last 10 → 2,453 clean / 85 blank
- Possible duplicates: same birth date + region + name start → 74 (deck says 78). Flag only, not merged

In [56]:
#18
city = customers["city"].str.strip().str.title()

aliases = {
    "La": "Los Angeles", "L.A.": "Los Angeles",
    "Sf": "San Francisco",
    "Chi": "Chicago", "Chi-Town": "Chicago",
    "N.Y.": "New York", "New York City": "New York",
    "Philly": "Philadelphia",
    "Sea": "Seattle", "Mia": "Miami", "Bos": "Boston",
    "Hou": "Houston", "Atl": "Atlanta", "Den": "Denver",
    "St Louis": "St. Louis",
}
customers["city_clean"] = city.replace(aliases)

print(customers["city"].nunique())
print(customers["city_clean"].nunique())

120
30


In [57]:
#19
email = customers["email"].fillna("").str.strip()
valid = email.str.match(r"^[^@\s]+@[^@\s]+\.[A-Za-z]+$")

customers["email_flag"] = np.where(email == "", "Blank", np.where(valid, "OK", "Invalid"))

print(customers["email_flag"].value_counts())

email_flag
OK         2298
Blank       219
Invalid      21
Name: count, dtype: int64


In [58]:
#20
digits = customers["phone"].fillna("").str.replace(r"\D", "", regex=True)
customers["phone_clean"] = digits.str[-10:]

print(digits.str.len().value_counts())
print(customers["phone_clean"].str.len().value_counts())

phone
10    1609
11     844
0       85
Name: count, dtype: int64
phone_clean
10    2453
0       85
Name: count, dtype: int64


In [59]:
#21
key = (customers["birth_date"].astype(str) + "|" + customers["region_name"] + "|"
       + customers["first_name"].str[0].str.upper() + "|"
       + customers["last_name"].str[:3].str.upper())

has_birth = customers["birth_date"].notna()
customers["possible_duplicate"] = has_birth & key.duplicated(keep=False)

print(customers["possible_duplicate"].sum())

74


## 9. Headline check (matches deck slide 46)
- Refunds added to each sales line; 804 lines returned
- Gross profit = revenue − cost − refunds
- 2023: 149,246 (21.1%) | 2024: 159,814 (20.7%) | 2025: 149,003 (17.2%)
- Refunds doubled in 2025 (19.7K → 39.7K)

In [60]:
#22
line_refunds = returns.groupby(["order_key", "sku_key"], as_index=False)["refund"].sum()
sales = sales.merge(line_refunds, on=["order_key", "sku_key"], how="left")
sales["refund"] = sales["refund"].fillna(0)
sales["returned"] = (sales["refund"] > 0).astype(int)
print(sales["returned"].sum())

summary = sales.groupby("order_year")[["net_revenue", "line_cost", "refund"]].sum()
summary["gross_profit"] = summary["net_revenue"] - summary["line_cost"] - summary["refund"]
summary["margin"] = summary["gross_profit"] / summary["net_revenue"]
print(summary.round(3))

804
            net_revenue  line_cost    refund  gross_profit  margin
order_year                                                        
2023          707009.75  541398.35  16365.25     149246.15   0.211
2024          773011.81  593495.20  19702.28     159814.33   0.207
2025          865952.69  677249.09  39700.43     149003.17   0.172


## 10. Hypothesis tests
- H1 Flagship: margin 19.6%, returns 3.0%, normal. z = 0.72 → REJECTED
- H2 Costs: flat at ~72% of gross sales → REJECTED
- H3 Online: both channels fell equally → REJECTED
- H4 Midwest discounts: 15.7% vs 5.7% → $20,349 → CONFIRMED
- H5 Kelvane: 50.7% returned, 84/113 quality, z = 10.56 → $16,054 → CONFIRMED
- What-if: 2025 profit $185,406 (+16% vs 2024) without the two problems

In [61]:
#23
stores = sales.groupby("store_name")[["net_revenue", "line_cost", "refund", "returned"]].sum()
stores["lines"] = sales.groupby("store_name").size()

stores["margin"] = (stores["net_revenue"] - stores["line_cost"] - stores["refund"]) / stores["net_revenue"]
stores["return_rate"] = stores["returned"] / stores["lines"]

print(stores.sort_values("net_revenue", ascending=False).head(4).round(3))

                             net_revenue  line_cost   refund  returned  lines  \
store_name                                                                      
New York Parkway Mall          241024.80  185383.89  8341.97        87   2911   
Denver Riverside Outlet         96276.67   73590.85  4386.64        45   1239   
Boston Riverside Mall           95206.58   73057.01  4303.15        43   1260   
Nashville Crossroads Street     94404.61   72264.57  3136.47        32   1113   

                             margin  return_rate  
store_name                                        
New York Parkway Mall         0.196        0.030  
Denver Riverside Outlet       0.190        0.036  
Boston Riverside Mall         0.187        0.034  
Nashville Crossroads Street   0.201        0.029  


In [62]:
#24
costs = sales.groupby("order_year")[["gross_sales", "line_cost"]].sum()
costs["cost_rate"] = costs["line_cost"] / costs["gross_sales"]
print(costs.round(3))

channel = sales.groupby(["order_year", "channel"])[["net_revenue", "line_cost", "refund"]].sum()
channel["margin"] = (channel["net_revenue"] - channel["line_cost"] - channel["refund"]) / channel["net_revenue"]
print(channel["margin"].unstack().round(3))

            gross_sales  line_cost  cost_rate
order_year                                   
2023          751506.98  541398.35      0.720
2024          825617.06  593495.20      0.719
2025          939686.23  677249.09      0.721
channel     Online  Store
order_year               
2023         0.192  0.216
2024         0.206  0.207
2025         0.168  0.174


In [63]:
#25
regions = sales.groupby(["region_name", "order_year"])[["net_revenue", "line_cost", "refund"]].sum()
regions["margin"] = (regions["net_revenue"] - regions["line_cost"] - regions["refund"]) / regions["net_revenue"]

print(regions["margin"].unstack().round(3))

order_year    2023   2024   2025
region_name                     
Midwest      0.215  0.186  0.105
Northeast    0.212  0.209  0.188
Southeast    0.199  0.213  0.195
Southwest    0.207  0.213  0.178
West         0.215  0.211  0.194


In [64]:
#26
s25 = sales[sales["order_year"] == 2025]
midwest = s25[s25["region_name"] == "Midwest"]
others = s25[s25["region_name"] != "Midwest"]

mw_rate = midwest["discount_amount"].sum() / midwest["gross_sales"].sum()
other_rate = others["discount_amount"].sum() / others["gross_sales"].sum()
excess_discount = midwest["gross_sales"].sum() * (mw_rate - other_rate)

print(round(mw_rate, 3), round(other_rate, 3), round(excess_discount))

0.157 0.057 20349


In [65]:
#27
appl = sales[(sales["category_name"] == "Home Appliances") & (sales["order_year"] == 2025)].copy()
appl["brand"] = appl["brand"].fillna("(no brand)")

brands = appl.groupby("brand")[["returned", "refund"]].sum()
brands["lines"] = appl.groupby("brand").size()
brands["return_rate"] = brands["returned"] / brands["lines"]
print(brands.sort_values("return_rate", ascending=False).round(3))

kel = returns[(returns["brand"] == "Kelvane") & (returns["order_year"] == 2025)]
print(kel["reason_group"].value_counts())

            returned    refund  lines  return_rate
brand                                             
Kelvane          113  17727.95    223        0.507
Hearthly          16   2113.79    373        0.043
Brisko             6    876.58    151        0.040
(no brand)         3    260.17    121        0.025
Oakhaven           1    229.49     91        0.011
reason_group
Quality    84
Other      29
Name: count, dtype: int64


In [66]:
#28
def two_prop_z(hits_1, n_1, hits_2, n_2):
    pooled = (hits_1 + hits_2) / (n_1 + n_2)
    se = np.sqrt(pooled * (1 - pooled) * (1 / n_1 + 1 / n_2))
    return (hits_1 / n_1 - hits_2 / n_2) / se

flag = sales[sales["store_name"] == "New York Parkway Mall"]
rest = sales[sales["store_name"] != "New York Parkway Mall"]
print(round(two_prop_z(flag["returned"].sum(), len(flag), rest["returned"].sum(), len(rest)), 2))

kelvane = sales[sales["brand"] == "Kelvane"]
k25 = kelvane[kelvane["order_year"] == 2025]
k24 = kelvane[kelvane["order_year"] == 2024]
print(round(two_prop_z(k25["returned"].sum(), len(k25), k24["returned"].sum(), len(k24)), 2))

0.72
10.56


In [67]:
#29
rate_25 = k25["returned"].mean()
rate_24 = k24["returned"].mean()
excess_refunds = k25["refund"].sum() * (1 - rate_24 / rate_25)

gp_24 = summary.loc[2024, "gross_profit"]
gp_25 = summary.loc[2025, "gross_profit"]
what_if = gp_25 + excess_discount + excess_refunds

print(round(excess_refunds), round(what_if), round(what_if / gp_24 - 1, 3))

16054 185406 0.16


## 11. Star schema export
- 5 dimensions (store, product, promotion, customer + GUEST, date)
- 2 facts (sales 28,917 rows, returns 804 rows)
- Saved to star_schema/ with a for loop

In [68]:
#30
import os
os.makedirs("star_schema", exist_ok=True)

sales["promo_code"] = sales["promo_code"].fillna("NONE")

dim_store = sales[["store_code", "store_name", "store_format", "store_city", "region_name"]].drop_duplicates(subset="store_code")
dim_product = sales[["sku", "product_name", "brand", "category_name", "department", "list_price"]].drop_duplicates(subset="sku")
dim_promotion = sales[["promo_code", "promo_name", "promo_type"]].drop_duplicates(subset="promo_code").fillna("No promotion")

print(len(dim_store), len(dim_product), len(dim_promotion))

30 597 60


In [69]:
#31
guest = pd.DataFrame([{"customer_code": "GUEST", "region_name": "n/a", "city_clean": "Guest"}])
dim_customer = pd.concat([
    customers[["customer_code", "gender", "birth_date", "city_clean", "region_name",
               "signup_date", "loyalty_tier", "email_flag", "possible_duplicate"]],
    guest,
])

dim_date = pd.DataFrame({"date": pd.date_range("2023-01-01", "2025-12-31", freq="D")})
dim_date["year"] = dim_date["date"].dt.year
dim_date["quarter"] = "Q" + dim_date["date"].dt.quarter.astype(str)
dim_date["month_no"] = dim_date["date"].dt.month
dim_date["month"] = dim_date["date"].dt.strftime("%b")

print(len(dim_customer), len(dim_date))

2539 1096


In [70]:
#32
fact_sales = sales[["order_number", "order_date", "store_code", "sku", "customer_code", "promo_code",
                    "channel", "payment_method", "quantity", "gross_sales", "discount_amount",
                    "net_revenue", "line_cost", "returned"]]

fact_returns = returns[["return_ref", "order_number", "order_date", "return_date", "store_code", "sku",
                        "qty_returned", "refund", "reason_group", "return_reason"]]

tables = {"dim_store": dim_store, "dim_product": dim_product, "dim_promotion": dim_promotion,
          "dim_customer": dim_customer, "dim_date": dim_date,
          "fact_sales": fact_sales, "fact_returns": fact_returns}

for name, table in tables.items():
    table.to_csv(f"star_schema/{name}.csv", index=False)
    print(name, len(table))

dim_store 30
dim_product 597
dim_promotion 60
dim_customer 2539
dim_date 1096
fact_sales 28917
fact_returns 804
